# SAADE–AUA v9.2 · FINAL FREEZE → HIDDEN_ONCE

**ОСТАТОЧНИЙ ЕТАП** сценарного оцінювання готових моделей: 69 незалежних REAL–TTS пар, три незмінні pretrained-детектори, усі 21 сценарій. **Без навчання, без повторення 876 330 прогнозів, без зміни оригінального AUA.**

Основа — перевірений результат **v9.1 repeat-padding** (повний експеримент) та методологія сценарного оцінювання [MAKE 2026, DOI 10.3390/make8060161](https://doi.org/10.3390/make8060161).

- **Етап 1, `FREEZE`:** перевіряє SHA всіх фінальних результатів v9.1 і фіксує протокол; **не відкриває hidden-аудіо**.
- **Етап 2, `HIDDEN_ONCE`:** після вашої явної зміни двох параметрів у конфігурації оцінює 69 hidden-пар **один раз**. Якщо середовище перерветься, дозволене **продовження того самого запуску** з SQLite без повторного інференсу готових комбінацій. Після `FINISHED` повторне тестування блокується.

**Важливо:** прихований тест містить **один TTS-голос (`lada`) із Piper**. Це незалежний голосовий holdout, **не незалежний TTS-рушій**. Перевірка узагальнення на інші генератори цим тестом не підтверджується.


## 0. Режим запуску (змінюємо лише тут)

In [ ]:
from pathlib import Path
PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT/'AUA'    # READ ONLY
RESULTS_ROOT = PROJECT/'results'

# ПЕРШИЙ запуск: FREEZE (не оцінює hidden).
# ДРУГИЙ запуск: HIDDEN_ONCE + підтвердження нижче.
RUN_PHASE = 'FREEZE'                 # 'FREEZE' або 'HIDDEN_ONCE'
HIDDEN_CONFIRMATION = 'NO'          # для HIDDEN_ONCE: 'I_AUTHORIZE_ONE_FINAL_HIDDEN_TEST'

BOOTSTRAP_REPS = 2000
SEED = 20261116
VERIFY_SHA256 = True
CACHE_MP3_TRANSFORMS = True

## 1. Файли завершеного v9.1 та перевірка незмінності

In [ ]:
import os, sys, gc, json, math, time, zipfile, hashlib, shutil, subprocess, tempfile, urllib.request, sqlite3
import datetime
from fractions import Fraction
from types import SimpleNamespace
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
from scipy import stats
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve, f1_score, confusion_matrix
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import display

assert RUN_PHASE in ('FREEZE','HIDDEN_ONCE'), 'Неприпустима фаза'
assert AUA.is_dir(), f'Не знайдено AUA: {AUA}'
RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
OLD_BASE=RESULTS_ROOT/'SAADE_AUA_2026_v9_COMPLETE_EVALUATION'
BASE=RESULTS_ROOT/'SAADE_AUA_2026_v9_1_AASIST_REPEAT_PADDING'
OUT=BASE/'FULL_EXPERIMENT'
FINAL=BASE/'FINAL_HIDDEN_v9_1'
MODEL_CACHE=OLD_BASE/'pretrained_model_cache'  # використовуємо вже перевірені ваги
assert OUT.is_dir() and MODEL_CACHE.is_dir(), f'Немає завершеного v9.1 / pretrained cache: {OUT}'
assert 'melnyk' not in str(FINAL).lower()
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'Потрібні ffmpeg і ffprobe'
V6_CANDIDATES=[RESULTS_ROOT/'SAADE_AUA_2026_v6_2_MATCHED_TTS',RESULTS_ROOT/'SAADE_AUA_2026_v6_MATCHED_TTS']
V6_DIR=next((p for p in V6_CANDIDATES if (p/'V6_LOCKED_MATCHED_SPLITS.csv').is_file()),None)
assert V6_DIR is not None,'Немає V6_LOCKED_MATCHED_SPLITS.csv'
MANIFEST_PATH=V6_DIR/'V6_LOCKED_MATCHED_SPLITS.csv'
SR=16000
WINDOW_SEC=4.0
WINDOW_N=int(SR*WINDOW_SEC)
MIN_TAIL_SEC=0.5
PROTOCOL_ID='SAADE-v9-20261116-full-canonical-all-segments-21-v8-compatible'
PREPROCESSING_ID='SAADE-v9.1-AASIST-repeat-active-audio-to-64600'
MODEL_NAMES=('AASIST_ONNX','AASIST_L_ONNX','WAV2VEC2_STAFFORD')

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()


def normhash(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()


def write_json_atomic(path,obj):
    tmp=Path(str(path)+'.part');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf8');os.replace(tmp,path)


def stable_seed(*parts):return int.from_bytes(hashlib.sha256('||'.join(map(str,parts)).encode()).digest()[:8],'big')%(2**32)


def audio_duration(path):
    try:
        x=sf.info(str(path));return float(x.duration)
    except Exception:
        proc=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],
                            capture_output=True,text=True,check=True,timeout=60)
        return float(proc.stdout.strip())


def read_wave(path,start_sec=0,active_n=WINDOW_N):
    active_n=int(active_n)
    if active_n<1 or active_n>WINDOW_N:raise ValueError('Некоректна активна довжина')
    try:
        with sf.SoundFile(str(path)) as f:
            rate=f.samplerate
            f.seek(max(0,int(round(float(start_sec)*rate))))
            wave=f.read(frames=int(math.ceil(WINDOW_SEC*rate)),dtype='float32',always_2d=True).mean(axis=1)
        if rate!=SR:
            p=Fraction(SR,rate);wave=signal.resample_poly(wave,p.numerator,p.denominator)
    except Exception:
        args=['ffmpeg','-nostdin','-v','error','-ss',str(max(0,float(start_sec))),'-i',str(path),'-t',str(WINDOW_SEC),
              '-ac','1','-ar',str(SR),'-f','f32le','-acodec','pcm_f32le','pipe:1']
        proc=subprocess.run(args,capture_output=True,timeout=180,check=True)
        wave=np.frombuffer(proc.stdout,dtype='<f4').copy()
    wave=np.nan_to_num(np.asarray(wave,dtype=np.float32),nan=0.,posinf=0.,neginf=0.)
    if not len(wave):raise ValueError('Порожнє декодоване аудіо: '+str(path))
    wave=wave[:active_n]
    return np.pad(wave,(0,max(0,WINDOW_N-len(wave))))[:WINDOW_N].astype(np.float32)

EXPECTED_FILE_SHA = {'00_COMPLETE_PROTOCOL.json': '5397f05974ccd6395d7e8e0d71fd9da824b9d9a4148ab4e357eeb8b96f05a78c', '02_SCENARIO_PROTOCOL.csv': '4733a85e8e6c9ab04d944725fd6ed1661a4c2610893ae47e90efbd648bd4760c', '03_MODEL_PROVENANCE.json': '383b70afd065f6acd5eaf0f23d8fe9df7c18c1fa968fdc5a2a95883914925636', '04_CALIBRATION.json': '82e3727e190bd7bc57a82db2ba8a4ffd70ec04be8da4857e35e16fe79d7374fa', '05_ALL_SEGMENT_PREDICTIONS.csv': '953fa51774e9837aa785b1ce9009bb2a12ff3361b57d807dad04adde58b68a27', '06_COMPLETED_SUMMARY.csv': 'd7d89acdd52fd7eb786bd6019a1fcd358cd3b5d539a897d9b3ef573bb69d0eb9', '10_FULL_EXPERIMENT_READINESS.json': '655419a8df86f61f9e7fd7c4570d9174f62c042db4cfd54b953402df01ae0fc6'}
PROTOCOL=json.loads((OUT/'00_COMPLETE_PROTOCOL.json').read_text(encoding='utf8'))
PROV=json.loads((OUT/'03_MODEL_PROVENANCE.json').read_text(encoding='utf8'))
CAL=json.loads((OUT/'04_CALIBRATION.json').read_text(encoding='utf8'))
READY=json.loads((OUT/'10_FULL_EXPERIMENT_READINESS.json').read_text(encoding='utf8'))
assert PROTOCOL['protocol']==PROTOCOL_ID
assert PROTOCOL['preprocessing']==PREPROCESSING_ID
assert PROTOCOL['n_total_segments']==13910
assert PROTOCOL['n_public_matched_segments']==888
assert PROTOCOL['n_aua_segments']==13022
assert PROTOCOL['scenario_hash'] and PROTOCOL['model_names']==list(MODEL_NAMES)
assert READY['actual_predictions']==READY['planned_predictions']==876330
assert READY['hidden_test_opened'] is False
assert READY['matched_hidden_pairs_protected']==69
assert READY['matched_public_pairs']==444
assert READY['model_training'] is False
assert sha256_file(MANIFEST_PATH)==PROTOCOL['dataset_sha'],'V6 matched manifest змінився від v9.1'

actual_file_sha={}
for fname,expected_sha in EXPECTED_FILE_SHA.items():
    fp=OUT/fname
    assert fp.is_file(),f'Не знайдено звіт: {fp}'
    dig=sha256_file(fp)
    assert dig==expected_sha, f'Вміст {fname} відрізняється від завершеного архіву v9.1. FREEZE зупинено.'
    actual_file_sha[fname]=dig

assert set(PROV)==set(CAL)==set(MODEL_NAMES)
for name in MODEL_NAMES:
    assert PROV[name]['weights_sha256']==CAL[name]['weights_sha256']==READY['model_sha'][name]
    assert CAL[name]['calibration_pairs']==110
    assert CAL[name]['no_model_training'] is True
    assert np.isfinite(float(CAL[name]['threshold']))
    if name!='WAV2VEC2_STAFFORD':
        assert PROV[name]['preprocessing']==PREPROCESSING_ID
        assert CAL[name]['protocol_sha']==sha256_file(OUT/'00_COMPLETE_PROTOCOL.json')
    else:
        assert CAL[name]['protocol_sha']==PROTOCOL['old_full_protocol_sha'], 'Неочікувана зміна старого Wav2Vec2 протоколу'

ONNX_CHECKPOINTS={
 'AASIST_ONNX':MODEL_CACHE/'aasist.onnx',
 'AASIST_L_ONNX':MODEL_CACHE/'aasist-l.onnx'
}
for name,fp in ONNX_CHECKPOINTS.items():
    assert fp.is_file(),f'Відсутній локальний ONNX: {fp}'
    assert sha256_file(fp)==PROV[name]['weights_sha256'],f'Змінилися ONNX-ваги: {name}'

assert not (OLD_BASE/'HIDDEN_ONCE_STARTED_v9.json').exists(), 'Старий v9 hidden уже запускався. Небезпека повторного тесту.'
assert not (OLD_BASE/'HIDDEN_ONCE_FINISHED_v9.json').exists(), 'Старий v9 hidden завершено. Новий запуск заборонено.'
assert not (OLD_BASE/'FROZEN_PROTOCOL_v9.json').exists(), 'Старий v9 заморожений: потрібен аудит протоколу перед новим hidden.'

if (OUT/'05_FULL_PREDICTIONS.sqlite3').is_file():
    ro=sqlite3.connect('file:'+str(OUT/'05_FULL_PREDICTIONS.sqlite3')+'?mode=ro',uri=True,timeout=30)
    try:
        observed=ro.execute('SELECT COUNT(*) FROM predictions').fetchone()[0]
        assert observed==876330,('Неповне SQLite v9.1',observed)
        for name in MODEL_NAMES:
            row=ro.execute('SELECT COUNT(*),MIN(checkpoint_sha),MAX(checkpoint_sha),MIN(threshold),MAX(threshold) FROM predictions WHERE model=?',(name,)).fetchone()
            assert row[0]==292110 and row[1]==row[2]==PROV[name]['weights_sha256']
            assert np.isclose(row[3],CAL[name]['threshold']) and np.isclose(row[4],CAL[name]['threshold'])
    finally:ro.close()

print('PASS: завершений v9.1 незмінний, 876 330 прогнозів, 3 checkpoint, 21 сценарій.')
print('Пороги тільки з train:',{m:round(float(CAL[m]['threshold']),7) for m in MODEL_NAMES})
print('Режим:',RUN_PHASE)

## 2. Точна реалізація 21 сценарію та препроцесингу v9.1

In [ ]:
SCENARIOS=[{'scenario':'clean','family':'clean','level':0,'parameter':None}]
for level,snr in enumerate([30,25,20,15,10],1):SCENARIOS.append({'scenario':f'noise_snr{snr}','family':'noise','level':level,'parameter':snr})
for level,kbps in enumerate([128,96,64,48,32],1):SCENARIOS.append({'scenario':f'mp3_{kbps}kbps','family':'mp3','level':level,'parameter':kbps})
for level,rt in enumerate([.15,.30,.45,.65,.85],1):SCENARIOS.append({'scenario':f'reverb_rt{rt:.2f}','family':'reverb','level':level,'parameter':rt})
for level,hz in enumerate([7500,6000,4500,3000,2000],1):SCENARIOS.append({'scenario':f'lowpass_{hz}Hz','family':'lowpass','level':level,'parameter':hz})
assert len(SCENARIOS)==21 and normhash(SCENARIOS)==PROTOCOL['scenario_hash']
assert set(pd.read_csv(OUT/'02_SCENARIO_PROTOCOL.csv').scenario)=={x['scenario'] for x in SCENARIOS}

def mp3_roundtrip(active,kbps):
    with tempfile.TemporaryDirectory(prefix='saade_mp3_') as tmp:
        wd=Path(tmp); src=wd/'original.wav'; mp3=wd/'compressed.mp3'; dec=wd/'decoded.wav'
        sf.write(src,active,SR,subtype='PCM_16')
        args=['ffmpeg','-nostdin','-y','-hide_banner','-loglevel','error']
        subprocess.run(args+['-i',str(src),'-c:a','libmp3lame','-b:a',f'{kbps}k',
                   '-ar',str(SR),'-ac','1',str(mp3)],check=True,timeout=60,stderr=subprocess.PIPE)
        subprocess.run(args+['-i',str(mp3),'-ar',str(SR),'-ac','1',str(dec)],
                   check=True,timeout=60,stderr=subprocess.PIPE)
        y,sr=sf.read(dec,dtype='float32',always_2d=True)
        assert sr==SR
        y=y.mean(axis=1)
    return np.pad(y[:len(active)],(0,max(0,len(active)-len(y))))[:len(active)]


def apply_scenario(wave,sc,pair_id,active_n=None):
    """Perturb ONLY active audio, keep deterministic equal padding per REAL/FAKE pair."""
    n=WINDOW_N if active_n is None else int(active_n)
    if n<1 or n>WINDOW_N:raise ValueError('Bad active_n')
    active=wave[:n].copy()
    family=sc['family']; val=sc['parameter']
    if family=='clean':out=active
    elif family=='noise':
        # ОДНАКОВИЙ шумовий шаблон для REAL та FAKE однієї пари, SNR залежить від RMS кожного сигналу.
        rng=np.random.default_rng(stable_seed(PROTOCOL_ID,pair_id,sc['scenario']))
        z=rng.standard_normal(n).astype(np.float32)
        z/=np.sqrt(np.mean(z.astype(np.float64)**2)+1e-12)
        rms=np.sqrt(np.mean(active.astype(np.float64)**2)+1e-12)
        out=np.clip(active+z*(rms/(10**(float(val)/20))),-1,1)
    elif family=='mp3':out=mp3_roundtrip(active,int(val))
    elif family=='reverb':
        # Синтетичний IR, НЕ емпірично виміряний RT60 кімнати.
        rt=float(val);m=int(min(SR*1.5,SR*rt*1.2))
        t=np.arange(m)/SR;env=np.exp(-6.9078*t/rt)
        rng=np.random.default_rng(stable_seed(PROTOCOL_ID,'IR',sc['scenario']))
        h=rng.normal(size=m).astype(np.float32)*env
        h[0]+=4.0
        h/=np.sqrt(np.sum(h.astype(np.float64)**2)+1e-12)
        out=np.clip(signal.fftconvolve(active,h,mode='full')[:n],-1,1)
    elif family=='lowpass':
        sos=signal.butter(6,float(val),fs=SR,output='sos',btype='lowpass')
        out=signal.sosfilt(sos,active)
    else:raise KeyError(family)
    out=np.asarray(out,dtype=np.float32)
    assert out.shape==(n,) and np.isfinite(out).all()
    return np.pad(out,(0,WINDOW_N-n)).astype(np.float32)


def transform_cached(wave,sc,row,cache_root):
    """Cache deterministic MP3 only; cache fingerprint includes source SHA and scenario.
    This cache is never packaged into the review archive.
    """
    active_n=int(row.active_n)
    if not CACHE_MP3_TRANSFORMS or sc['family']!='mp3':
        return apply_scenario(wave,sc,str(row.pair_id if row.pair_id else row.sample_id),active_n)
    key=normhash({'protocol':PROTOCOL_ID,'source_sha':str(row.expected_sha),
                  'sample_id':str(row.sample_id),'label':int(row.label),
                  'scenario':sc['scenario'],'active_n':active_n})
    dest=Path(cache_root)/(key+'.npy')
    if dest.is_file():
        arr=np.load(dest,allow_pickle=False)
        if arr.shape!=(WINDOW_N,) or arr.dtype!=np.float32 or not np.isfinite(arr).all():
            raise RuntimeError('Пошкоджено кеш MP3: '+str(dest))
        return arr
    out=apply_scenario(wave,sc,str(row.pair_id if row.pair_id else row.sample_id),active_n)
    dest.parent.mkdir(parents=True,exist_ok=True)
    tmp=dest.with_name(dest.name+'.part.npy')
    np.save(tmp,out,allow_pickle=False)
    os.replace(tmp,dest)
    return out

probe=np.sin(2*np.pi*440*np.arange(WINDOW_N)/SR).astype(np.float32)*0.05
for scenario in SCENARIOS:
    waveform=apply_scenario(probe,scenario,'safe-control',SR*3)
    assert waveform.shape==(WINDOW_N,) and np.isfinite(waveform).all()
    assert np.all(waveform[SR*3:]==0)
assert np.array_equal(apply_scenario(probe,SCENARIOS[1],'test-pair',SR*3),
                      apply_scenario(probe,SCENARIOS[1],'test-pair',SR*3))
print('PASS: 21 сценарій і стабільні deterministic seeds збігаються з v9.1')

## 3. Ті самі pretrained-детектори та орієнтація FAKE/REAL

In [ ]:
MODEL_SOURCES={
 'AASIST_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST','file':'aasist.onnx',
                'source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST'},
 'AASIST_L_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST-L','file':'aasist-l.onnx',
                'source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L'},
 'WAV2VEC2_STAFFORD':{'kind':'hf','repo':'garystafford/wav2vec2-deepfake-voice-detector',
                     'expected_fake_index':1,'source':'https://huggingface.co/garystafford/wav2vec2-deepfake-voice-detector'}
}
SELECTED_MODELS=list(MODEL_NAMES)

def download_identity(repo,filename,dest,min_bytes=50000):
    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.is_file() and dest.stat().st_size>=min_bytes:return dest
    url=f'https://huggingface.co/{repo}/resolve/main/{filename}'
    req=urllib.request.Request(url,headers={'User-Agent':'SAADE-v8-research/1.0','Accept-Encoding':'identity'})
    part=dest.with_name(dest.name+'.part')
    try:
        with urllib.request.urlopen(req,timeout=180) as response, open(part,'wb') as f:
            while True:
                b=response.read(2*1024*1024)
                if not b:break
                f.write(b)
        if part.stat().st_size<min_bytes or part.open('rb').read(32).lstrip().lower().startswith(b'<html'):
            raise RuntimeError('Файл моделі не завантажено повністю: '+url)
        os.replace(part,dest)
    finally:part.unlink(missing_ok=True)
    return dest


def fake_index_from_labels(id2label,expected):
    labels={int(k):str(v).strip().lower() for k,v in id2label.items()}
    assert set(labels)=={0,1},'Не 2 класи: '+repr(labels)
    explicit=[k for k,v in labels.items() if 'fake' in v or 'spoof' in v or 'synthetic' in v]
    if len(explicit)==1 and explicit[0]!=expected:
        raise RuntimeError('Класи моделі не збігаються з документацією: '+str(labels))
    if len(explicit)==0 and set(labels.values())!={'label_0','label_1'}:
        raise RuntimeError('Неможливо підтвердити мітки FAKE/REAL: '+str(labels))
    if expected not in labels:raise ValueError('Неправильний FAKE index')
    return expected,labels


class OnnxAASIST:
    def __init__(self,name,cfg):
        import onnxruntime as ort
        self.name=name;self.source=cfg['source']
        fp=download_identity(cfg['repo'],cfg['file'],MODEL_CACHE/cfg['file'])
        self.weights_sha256=sha256_file(fp)
        self.model_revision=self.weights_sha256
        self.session=ort.InferenceSession(str(fp),providers=([ 'CUDAExecutionProvider','CPUExecutionProvider' ] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']))
        ins=self.session.get_inputs()
        assert len(ins)==1,'Непідтримуваний ONNX input'
        self.input_name=ins[0].name
        self.input_shape=ins[0].shape
        self.score_kind='negative_bonafide_logit (higher is fake, not probability)'
        self.labels={0:'spoof',1:'bonafide'}  # реальний порядок логітів, підтверджений аудитом ONNX ↔ PyTorch
    def score_fake(self,w,active_n=WINDOW_N):
        # Еталон AASIST: повторювати ДІЙСНИЙ активний сигнал до 64600 семплів.
        # Повторення zero-padding було б методологічно помилковим для коротких хвостів.
        n=int(active_n)
        assert 1 <= n <= WINDOW_N, f'Неправильна активна довжина: {n}'
        active=np.asarray(w[:n],dtype=np.float32)
        assert active.shape==(n,) and np.isfinite(active).all()
        x=np.tile(active, int(math.ceil(64600/n)))[:64600].astype(np.float32)
        assert x.shape==(64600,)
        sh=self.input_shape
        if len(sh)==2:inp=x[None,:]
        elif len(sh)==3 and sh[1] in (1,'1',None):inp=x[None,None,:]
        else:raise RuntimeError('Невідомий shape ONNX: '+str(sh))
        outs=self.session.run(None,{self.input_name:inp})
        vals=np.asarray(outs[0],dtype=float).reshape(-1)
        if vals.size==1:bonafide=vals[0]
        elif vals.size==2:bonafide=vals[1]
        else:raise RuntimeError('Непередбачений вихід AASIST: '+str(vals.shape))
        return float(-bonafide)
    def close(self):del self.session


def locate_hf_weights(repo,revision=None,fallback=None):
    """Find exact cached weights to fingerprint. No unknown revision is accepted."""
    candidates=[]
    try:
        from huggingface_hub import constants
        cache=Path(constants.HF_HUB_CACHE)
        root=cache/('models--'+repo.replace('/','--'))/'snapshots'
        if revision and revision!='unknown':
            q=root/revision/'model.safetensors'
            if q.exists():candidates.append(q)
        else:
            candidates.extend(root.glob('*/model.safetensors'))
    except Exception:pass
    if fallback and Path(fallback).exists():candidates.insert(0,Path(fallback))
    # Multiple ambiguous snapshots MUST NOT be chosen using mtime alone.
    hashes={sha256_file(p):p for p in candidates if p.is_file() and p.stat().st_size>50_000}
    if len(hashes)!=1:
        raise RuntimeError(f'Неможливо однозначно зафіксувати SHA-256 checkpoint {repo}. '
                           f'Знайдено варіантів: {len(hashes)}. Приберіть зайві версії кешу або '
                           'завантажте модель в окремий локальний каталог.')
    return next(iter(hashes)), str(next(iter(hashes.values())))


class HFWav2Vec2:
    def __init__(self,name,cfg):
        import torch
        from transformers import AutoFeatureExtractor,AutoModelForAudioClassification
        self.name=name;self.source=cfg['source'];self.torch=torch
        self.device='cuda' if torch.cuda.is_available() else 'cpu'
        self._cache_weights=None
        try:
            self.fe=AutoFeatureExtractor.from_pretrained(cfg['repo'],trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(cfg['repo'],trust_remote_code=False)
        except Exception as exc:
            print('HF download problem; HTTPS identity fallback:',str(exc)[:180])
            folder=MODEL_CACHE/name
            for f in ['config.json','preprocessor_config.json','model.safetensors']:
                download_identity(cfg['repo'],f,folder/f,50000 if f.endswith('safetensors') else 100)
            self._cache_weights=folder/'model.safetensors'
            self.fe=AutoFeatureExtractor.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False)
        self.fake_idx,self.labels=fake_index_from_labels(self.model.config.id2label,cfg['expected_fake_index'])
        commit=getattr(self.model.config,'_commit_hash',None)
        self.weights_sha256,self.weights_file=locate_hf_weights(cfg['repo'],commit,self._cache_weights)
        self.model_revision=self.weights_sha256
        self.model=self.model.to(self.device).eval()
        self.model.requires_grad_(False)
        self.score_kind='softmax_fake_probability'
    def score_fake(self,w):
        batch=self.fe(w,sampling_rate=SR,return_tensors='pt',padding=True)
        batch={k:v.to(self.device) for k,v in batch.items()}
        with self.torch.inference_mode():
            probs=self.torch.softmax(self.model(**batch).logits,dim=-1)
        return float(probs[0,self.fake_idx].item())
    def close(self):
        del self.model
        if self.device=='cuda':self.torch.cuda.empty_cache()


def load_detector(name):
    cfg=MODEL_SOURCES[name]
    return OnnxAASIST(name,cfg) if cfg['kind']=='onnx' else HFWav2Vec2(name,cfg)


def _wav2vec2_score_many(self,waves):
    tensors=self.fe([np.asarray(w,dtype=np.float32) for w in waves],sampling_rate=SR,return_tensors='pt',padding=True)
    tensors={k:v.to(self.device) for k,v in tensors.items()}
    with self.torch.inference_mode():
        probs=self.torch.softmax(self.model(**tensors).logits,dim=-1)
    return [float(x) for x in probs[:,self.fake_idx].detach().cpu().numpy()]

HFWav2Vec2.score_fake_many=_wav2vec2_score_many

# Інференс НЕ МОЖНА виконати, доки не зафіксовано й повторно не звірено checkpoint.
def load_frozen_detector(name):
    d=load_detector(name)
    assert d.weights_sha256==PROV[name]['weights_sha256'], f'Checkpoint {name} не той, що у v9.1!'
    assert d.score_kind==PROV[name]['score_kind'], f'Інтерпретація score змінилася: {name}'
    assert {str(k):str(v) for k,v in d.labels.items()}=={str(k):str(v) for k,v in PROV[name]['id2label'].items()}
    return d

q=np.linspace(-.1,.1,101,dtype=np.float32)
for n in (24,301,64000):
    active=np.resize(q,n)
    repeat=np.tile(active,math.ceil(64600/n))[:64600]
    assert len(repeat)==64600 and np.array_equal(repeat[:n],active)
print('PASS: pretrained configuration; AASIST repeat-padding перевірено без читання hidden')

## 4. ЗАМОРОЖУВАННЯ: точна копія повного результату v9.1

In [ ]:
# Фіксуємо конкретний стан v9.1, всі пороги та незмінність маніфесту.
# У FREEZE не читаємо ані приховані шляхи, ані WAV.
FINAL.mkdir(parents=True,exist_ok=True)
FREEZE_FILE=FINAL/'FROZEN_PROTOCOL_v9_1.json'
START_FILE=FINAL/'HIDDEN_ONCE_STARTED_v9_1.json'
FINISH_FILE=FINAL/'HIDDEN_ONCE_FINISHED_v9_1.json'
RESULT_DB=FINAL/'HIDDEN_ONCE_PREDICTIONS.sqlite3'
FINAL_OUTPUT=RESULTS_ROOT/'SAADE_AUA_v9_2_FINAL_HIDDEN_ONCE_FOR_REVIEW.zip'

PRIVATE_WORK=OUT/'01_PRIVATE_FULL_WORKLIST.csv'
assert PRIVATE_WORK.is_file(),'Не знайдено приватний worklist v9.1'

LOCK_BODY={
    'schema':'SAADE-v9.2-hidden-once-immutable-lock',
    'source_protocol_sha':actual_file_sha['00_COMPLETE_PROTOCOL.json'],
    'v9_1_artifact_sha':actual_file_sha,
    'full_worklist_sha':sha256_file(PRIVATE_WORK),
    'v6_manifest_sha':sha256_file(MANIFEST_PATH),
    'scenarios_hash':PROTOCOL['scenario_hash'],
    'preprocessing':PREPROCESSING_ID,
    'model_sha':{m:PROV[m]['weights_sha256'] for m in MODEL_NAMES},
    'model_score_kind':{m:PROV[m]['score_kind'] for m in MODEL_NAMES},
    'thresholds':{m:float(CAL[m]['threshold']) for m in MODEL_NAMES},
    'n_hidden_pairs':69,
    'seed':SEED,
    'n_scenarios':21,
    'n_models':3,
    'n_expected_hidden_scores':69*2*21*3,
    'no_model_training':True,
    'validation_fake_voice_count':1,
    'hidden_voice_count_expected':1,
    'notes':'Holdout TTS speaker; not independent TTS generator'
}
LOCK_SHA=normhash(LOCK_BODY)

def verify_frozen():
    assert FREEZE_FILE.is_file(),'Протокол ще не заморожено. Спершу RUN_PHASE=FREEZE.'
    saved=json.loads(FREEZE_FILE.read_text(encoding='utf8'))
    assert saved['lock_body']==LOCK_BODY and saved['lock_sha']==LOCK_SHA, 'Протокол змінився після FREEZE!'
    return saved

if RUN_PHASE=='FREEZE':
    assert HIDDEN_CONFIRMATION=='NO','На етапі FREEZE підтвердження hidden має бути NO'
    assert not START_FILE.exists() and not FINISH_FILE.exists(),'Hidden уже починався. Повторно FREEZE заборонено.'
    if FREEZE_FILE.exists():
        verify_frozen()
        print('FREEZE вже виконано; стан незмінний:',LOCK_SHA)
    else:
        saved={'lock_sha':LOCK_SHA,'lock_body':LOCK_BODY,
            'frozen_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
        with FREEZE_FILE.open('x',encoding='utf8') as f:
            json.dump(saved,f,indent=2,ensure_ascii=False)
        print('FREEZE УСПІШНИЙ. Hidden НЕ відкривався.')
        print('Файл:',FREEZE_FILE)
    print('Для одноразового hidden: RUN_PHASE="HIDDEN_ONCE", HIDDEN_CONFIRMATION="I_AUTHORIZE_ONE_FINAL_HIDDEN_TEST", потім Run All.')
else:
    assert HIDDEN_CONFIRMATION=='I_AUTHORIZE_ONE_FINAL_HIDDEN_TEST','Потрібне явне підтвердження одноразового hidden test'
    verify_frozen()
    if FINISH_FILE.exists():
        raise RuntimeError('HIDDEN_ONCE вже завершено. Повторний запуск заборонений.')
    print('LOCK звірено. Всі checkpoint і пороги незмінні:',LOCK_SHA)

## 5. Одноразовий hidden test (виконується ТІЛЬКИ в режимі `HIDDEN_ONCE`)

In [ ]:
if RUN_PHASE!='HIDDEN_ONCE':
    print('FREEZE завершено. Hidden WAV і мітки не читалися.')
else:
    # Читання hidden-маніфесту тільки ПІСЛЯ FREEZE.
    raw=pd.read_csv(MANIFEST_PATH,low_memory=False)
    required={'path','pair_id','label','split','group_id','tts_voice_id','working_sha256'}
    assert required.issubset(raw.columns),f'Немає колонок маніфесту: {required-set(raw.columns)}'
    raw['label']=pd.to_numeric(raw.label,errors='raise').astype(int)
    raw['pair_id']=raw.pair_id.astype(str)
    assert raw.label.isin([0,1]).all()
    assert raw.groupby('pair_id').split.nunique().max()==1
    assert raw.groupby(['pair_id','label']).size().eq(1).all()
    assert raw.groupby('pair_id').label.nunique().eq(2).all()
    for cls in (0,1):
        assert raw.loc[raw.label.eq(cls)].groupby('group_id').split.nunique().max()==1, 'Виявлено голосовий leakage між split'
    public=raw.loc[raw.split.ne('hidden_test')].copy()
    hidden=raw.loc[raw.split.eq('hidden_test')].copy()
    assert hidden.pair_id.nunique()==69 and len(hidden)==138, 'Змінилася кількість hidden-пар'
    assert public.pair_id.nunique()==444
    for split,n in [('train',110),('adaptation',240),('validation',94)]:
        assert public.loc[public.split.eq(split),'pair_id'].nunique()==n
    assert hidden.loc[hidden.label.eq(1),'tts_voice_id'].nunique()==1, 'Неочікувана зміна кількості TTS-голосів'
    assert set(hidden.loc[hidden.label.eq(1),'tts_voice_id'])=={'lada'}, 'Hidden-голос не збігається з зафіксованим'
    assert set(hidden.pair_id).isdisjoint(set(public.pair_id))
    if 'parent_file_id' in raw:
        assert raw.dropna(subset=['parent_file_id']).groupby('parent_file_id').split.nunique().max()<=1
    for col in ('text_key','input_text_id','working_sha256'):
        if col in raw:
            s=raw.dropna(subset=[col]);s=s.loc[s[col].astype(str).str.strip().ne('')]
            assert s.groupby(col).split.nunique().max()<=1,f'Можливий перетин {col} між вибірками'

    receipts_path=V6_DIR/'V6_TTS_RECEIPTS.csv'
    assert receipts_path.is_file()
    receipts=pd.read_csv(receipts_path,low_memory=False)
    assert {'pair_id','generated_path','sha256'}.issubset(receipts.columns)
    rec_lookup={(str(r.pair_id),str(r.generated_path)):str(r.sha256).lower()
                for r in receipts.itertuples(index=False)}
    work=pd.read_csv(PRIVATE_WORK,low_memory=False,keep_default_na=False)
    assert 'path' in work and 'expected_sha' in work
    public_paths={str(Path(s).resolve()) for s in work.path.astype(str)}
    public_sha={str(x).lower() for x in work.expected_sha.astype(str) if len(str(x))==64}

    prepared=[]
    checked_hash={}
    for pid,g in hidden.groupby('pair_id',sort=True):
        seconds=[]
        for r in g.itertuples(index=False):
            fpath=Path(str(r.path))
            assert fpath.is_file(),f'Не знайдено hidden WAV: {pid}, label={r.label}'
            assert str(fpath.resolve()) not in public_paths,'Hidden SHA/path leakage у відкриту вибірку'
            expected=(rec_lookup.get((str(r.pair_id),str(r.path))) if int(r.label)==1
                      else str(getattr(r,'working_sha256','')).lower())
            assert expected and len(expected)==64,f'Невідомий SHA hidden: {pid}/{r.label}'
            assert expected not in public_sha,f'Hidden SHA присутній у відкритій вибірці: {pid}'
            if VERIFY_SHA256:
                dig=checked_hash.get(str(fpath))
                if dig is None:
                    dig=sha256_file(fpath)
                    checked_hash[str(fpath)]=dig
                assert dig==expected,f'Не збігається SHA hidden WAV: {pid}/{r.label}'
            start=float(getattr(r,'segment_start_sec',0) or 0)
            duration=audio_duration(fpath)-start
            assert duration>=MIN_TAIL_SEC,(pid,int(r.label),duration)
            seconds.append(duration)
            prepared.append(dict(pair_id=str(pid),label=int(r.label),path=str(fpath),start_sec=start,
                                 expected_sha=expected,tts_voice_id=str(r.tts_voice_id)))
        common_active_n=int(min(WINDOW_N,max(1,math.floor(min(seconds)*SR))))
        for row in prepared[-2:]:
            assert row['pair_id']==str(pid)
            row['active_n']=common_active_n
            row['sample_id']='hidden:'+row['pair_id']+':'+str(row['label'])
    assert len(prepared)==138 and len({x['sample_id'] for x in prepared})==138

    # Ініціалізуємо ТІ САМІ pretrained-моделі і перевіряємо SHA ДО створення STARTED.
    loaded={}
    try:
        for name in MODEL_NAMES:
            loaded[name]=load_frozen_detector(name)
        print('Всі pretrained checkpoint успішно звірені; починаємо єдиний hidden-запуск.')

        FINAL.mkdir(parents=True,exist_ok=True)
        if START_FILE.exists():
            state=json.loads(START_FILE.read_text(encoding='utf8'))
            assert state['lock_sha']==LOCK_SHA and state['expected_predictions']==LOCK_BODY['n_expected_hidden_scores']
            assert not FINISH_FILE.exists(), 'Hidden test уже завершено; повтор заборонено'
            print('Технічне відновлення ТОГО САМОГО одноразового запуску. Уже обчислені комбінації пропускаються.')
        else:
            with START_FILE.open('x',encoding='utf8') as f:
                json.dump({'lock_sha':LOCK_SHA,'started_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
                           'expected_predictions':LOCK_BODY['n_expected_hidden_scores'],
                           'resume_allowed_only_without_retesting':True},f,ensure_ascii=False,indent=2)

        conn=sqlite3.connect(RESULT_DB,timeout=60)
        try:
            conn.execute('PRAGMA journal_mode=WAL')
            conn.execute('PRAGMA synchronous=FULL')
            conn.execute("""CREATE TABLE IF NOT EXISTS predictions (
                model TEXT NOT NULL,pair_id TEXT NOT NULL,label INTEGER NOT NULL,sample_id TEXT NOT NULL,
                scenario TEXT NOT NULL,family TEXT NOT NULL,level INTEGER NOT NULL,
                score_fake REAL NOT NULL,threshold REAL NOT NULL,pred_label INTEGER NOT NULL,
                checkpoint_sha TEXT NOT NULL,source_sha TEXT NOT NULL,
                PRIMARY KEY(model,sample_id,scenario))""")
            conn.commit()
            # Не змішуємо жодні результати інших протоколів.
            state_rows=conn.execute('SELECT DISTINCT model,checkpoint_sha,threshold FROM predictions').fetchall()
            for nm,digest,th in state_rows:
                assert nm in MODEL_NAMES and digest==PROV[nm]['weights_sha256']
                assert float(th)==float(CAL[nm]['threshold'])
            total_new=0;total_reused=0
            for model in MODEL_NAMES:
                detector=loaded[model]
                threshold=float(CAL[model]['threshold'])
                print('HIDDEN:',model,'threshold:',threshold)
                for k,row in enumerate(prepared,1):
                    complete=conn.execute('SELECT COUNT(*) FROM predictions WHERE model=? AND sample_id=?',
                        (model,row['sample_id'])).fetchone()[0]
                    assert complete in (0,len(SCENARIOS)),f'Частковий результат у SQLite: {model}/{row["sample_id"]}'
                    if complete==len(SCENARIOS):
                        total_reused+=complete
                        continue
                    sound=read_wave(row['path'],row['start_sec'],row['active_n'])
                    entry=SimpleNamespace(**row)
                    transformed=[transform_cached(sound,sc,entry,FINAL/'_mp3_cache') for sc in SCENARIOS]
                    if hasattr(detector,'score_fake_many'):
                        predictions=[]
                        for j in range(0,len(transformed),4):
                            predictions.extend(detector.score_fake_many(transformed[j:j+4]))
                    else:
                        predictions=[detector.score_fake(w,active_n=row['active_n']) for w in transformed]
                    assert len(predictions)==21 and np.isfinite(predictions).all()
                    records=[]
                    for sc,score in zip(SCENARIOS,predictions):
                        records.append((model,row['pair_id'],row['label'],row['sample_id'],sc['scenario'],
                            sc['family'],int(sc['level']),float(score),threshold,int(float(score)>=threshold),
                            PROV[model]['weights_sha256'],row['expected_sha']))
                    with conn:
                        conn.executemany("INSERT INTO predictions VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",records)
                    total_new+=21
                    if k%20==0:
                        print(f' {model}: {k}/138 аудіозаписів; нових {total_new:,}; використано кеш {total_reused:,}')
            n_total=conn.execute('SELECT COUNT(*) FROM predictions').fetchone()[0]
            assert n_total==LOCK_BODY['n_expected_hidden_scores'],f'Неповний hidden: {n_total}/{LOCK_BODY["n_expected_hidden_scores"]}'
            hidden_predictions=pd.read_sql_query('SELECT * FROM predictions ORDER BY model,sample_id,scenario',conn)
        finally:
            conn.close()
    finally:
        for detector in loaded.values():
            try:detector.close()
            except Exception:pass
        gc.collect()

    print('HIDDEN_ONCE: 100% покриття:',len(hidden_predictions),'прогнозів. Нових:',total_new,'з кешу:',total_reused)

## 6. Статистика, порівняння validation ↔ hidden, архів для статті

In [ ]:
def eer_score(y,score):
    fpr,tpr,_=roc_curve(y,score)
    fnr=1-tpr;delta=fpr-fnr
    cross=np.flatnonzero(delta[:-1]*delta[1:]<=0)
    if len(cross):
        i=int(cross[0]);d=delta[i+1]-delta[i]
        v=0 if abs(d)<1e-15 else -delta[i]/d
        return float(fpr[i]+v*(fpr[i+1]-fpr[i]))
    return float((fpr[np.argmin(abs(delta))]+fnr[np.argmin(abs(delta))])/2)


def binary_metrics(df):
    y=df.label.to_numpy(int);p=df.pred_label.to_numpy(int);scores=df.score_fake.to_numpy(float)
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    ans={'n_records':len(df),'n_real':int((y==0).sum()),'n_fake':int((y==1).sum()),
         'TP':int(tp),'TN':int(tn),'FP':int(fp),'FN':int(fn),
         'TPR_fake':float(tp/(tp+fn)) if tp+fn else np.nan,
         'TNR_real':float(tn/(tn+fp)) if tn+fp else np.nan,
         'F1_fake':float(f1_score(y,p,zero_division=0)) if (y==1).any() else np.nan}
    if set(y)=={0,1}:
        ans.update(BA=float(balanced_accuracy_score(y,p)),ROC_AUC=float(roc_auc_score(y,scores)),EER=float(eer_score(y,scores)))
    else:ans.update(BA=np.nan,ROC_AUC=np.nan,EER=np.nan)
    return ans

if RUN_PHASE!='HIDDEN_ONCE':
    print('FREEZE завершено. Для hidden змініть дві змінні в комірці 0 та виконайте Run All.')
else:
    metrics_rows=[]
    for (model,scenario,family,level),g in hidden_predictions.groupby(['model','scenario','family','level'],sort=True):
        metrics_rows.append({'model':model,'scenario':scenario,'family':family,'level':int(level),
                             **binary_metrics(g)})
    hmetrics=pd.DataFrame(metrics_rows)
    hsummary=[]
    for name,subset in hmetrics.groupby('model'):
        clean=subset.loc[subset.family.eq('clean')].iloc[0]
        cat=subset.loc[subset.family.ne('clean')].groupby('family').BA.mean()
        hsummary.append({'model':name,'hidden_pairs':69,'hidden_voice':'lada',
                         'clean_BA':float(clean.BA),'clean_ROC_AUC':float(clean.ROC_AUC),
                         'ASRI_Audio':float(cat.mean()),'worst_stress_BA':float(subset.loc[subset.family.ne('clean')].BA.min()),
                         'scope':'single held-out Piper voice; cross-generator generalization unsupported'})
    hsummary=pd.DataFrame(hsummary).sort_values('ASRI_Audio',ascending=False)

    # Парний bootstrap по 69 pair_id; не вважати 21 сценарій незалежними спостереженнями.
    pairs=sorted(hidden_predictions.pair_id.unique())
    assert len(pairs)==69 and hidden_predictions.groupby(['model','scenario','pair_id']).label.nunique().eq(2).all()
    scenarios_stress=[s['scenario'] for s in SCENARIOS if s['family']!='clean']
    rng=np.random.default_rng(SEED)
    indices=rng.integers(0,len(pairs),size=(BOOTSTRAP_REPS,len(pairs)))
    per_model={}
    for model in MODEL_NAMES:
        subset=hidden_predictions.loc[(hidden_predictions.model.eq(model)) & (hidden_predictions.scenario.isin(scenarios_stress))]
        pvt=subset.pivot(index=['pair_id','scenario'],columns='label',values='pred_label')
        pvt=pvt.reindex(pd.MultiIndex.from_product([pairs,scenarios_stress],names=['pair_id','scenario']))
        assert pvt.notna().all().all()
        tnr=(1-pvt[0].to_numpy().reshape(len(pairs),len(scenarios_stress)))
        tpr=pvt[1].to_numpy().reshape(len(pairs),len(scenarios_stress))
        per_pair=(tnr+tpr)/2
        per_model[model]=per_pair.mean(axis=1)
    boots={model:per_pair[indices].mean(axis=1) for model,per_pair in per_model.items()}
    ci=[]
    for name in MODEL_NAMES:
        lo,hi=np.quantile(boots[name],[.025,.975])
        ci.append({'model':name,'ASRI_Audio':float(per_model[name].mean()),
                   'CI95_lower':float(lo),'CI95_upper':float(hi),'bootstrap_unit':'matched pair_id',
                   'n_pairs':69,'n_fake_voices':1})
    ci=pd.DataFrame(ci)
    diffs=[]
    for i,a in enumerate(MODEL_NAMES):
        for b in MODEL_NAMES[i+1:]:
            delta=boots[a]-boots[b]
            lo,hi=np.quantile(delta,[.025,.975])
            adj_lo,adj_hi=np.quantile(delta,[0.05/(2*3),1-0.05/(2*3)])
            diffs.append({'model_a':a,'model_b':b,'ASRI_delta':float(per_model[a].mean()-per_model[b].mean()),
                          'CI95_lower':float(lo),'CI95_upper':float(hi),
                          'significant_95_nominal':bool(lo>0 or hi<0),
                          'CI98_33_lower_bonferroni':float(adj_lo),'CI98_33_upper_bonferroni':float(adj_hi),
                          'significant_bonferroni_3_comparisons':bool(adj_lo>0 or adj_hi<0),'n_pairs':69})
    diff_df=pd.DataFrame(diffs)

    # Пряме зіставлення зі збереженою validation того самого v9.1.
    val_summary=pd.read_csv(OUT/'06_COMPLETED_SUMMARY.csv')
    val_summary=val_summary.loc[val_summary.cohort.eq('MATCHED_VALIDATION'),['model','clean_BA','ASRI_Audio']]
    comparison=hsummary.merge(val_summary,on='model',suffixes=('_hidden','_validation'))
    comparison['ASRI_shift_hidden_minus_validation']=comparison.ASRI_Audio_hidden-comparison.ASRI_Audio_validation

    # Зберігаємо тільки остаточні результати у створеному каталозі, без вихідних WAV та шляхів.
    hdir=FINAL/'RESULTS';hdir.mkdir(exist_ok=True)
    hidden_predictions.to_csv(hdir/'01_HIDDEN_PREDICTIONS.csv',index=False)
    hmetrics.to_csv(hdir/'02_HIDDEN_SCENARIO_METRICS.csv',index=False)
    hsummary.to_csv(hdir/'03_HIDDEN_ASRI_SUMMARY.csv',index=False)
    ci.to_csv(hdir/'04_HIDDEN_PAIR_BOOTSTRAP_CI.csv',index=False)
    diff_df.to_csv(hdir/'05_HIDDEN_PAIRWISE_DIFFERENCES.csv',index=False)
    comparison.to_csv(hdir/'06_VALIDATION_VS_HIDDEN.csv',index=False)
    readiness={'status':'COMPLETE','protocol':'v9.1 repeat-padding, frozen','freeze_sha':LOCK_SHA,
               'n_pairs':69,'n_models':3,'n_scenarios':21,'n_predictions':len(hidden_predictions),
               'checkpoint_sha':{k:PROV[k]['weights_sha256'] for k in MODEL_NAMES},
               'thresholds':{k:float(CAL[k]['threshold']) for k in MODEL_NAMES},
               'single_hidden_tts_voice':'lada','cross_generator_generalization':False,
               'no_training':True,'reproducibility_note':'No test-time threshold/model selection permitted'}
    write_json_atomic(hdir/'00_FINAL_REPORT.json',readiness)
    assert not FINISH_FILE.exists(),'Hidden уже завершено; не створюємо другу версію результатів'
    with zipfile.ZipFile(FINAL_OUTPUT,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for f in sorted(hdir.iterdir()):
            if f.is_file():z.write(f,arcname=f.name)
        z.write(FREEZE_FILE,arcname='FROZEN_PROTOCOL_v9_1.json')
    assert FINAL_OUTPUT.is_file() and FINAL_OUTPUT.stat().st_size>5000
    with FINISH_FILE.open('x',encoding='utf8') as f:
        json.dump({'lock_sha':LOCK_SHA,'finished_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
                   'n_predictions':len(hidden_predictions),'zip_sha256':sha256_file(FINAL_OUTPUT)},f,ensure_ascii=False,indent=2)
    # Можемо безпечно очистити ТІЛЬКИ власний тимчасовий кеш після 100% завершення.
    if (FINAL/'_mp3_cache').is_dir():shutil.rmtree(FINAL/'_mp3_cache')
    print('HIDDEN_ONCE ЗАВЕРШЕНО; повторний запуск заблоковано. Звіт:',FINAL_OUTPUT)
    print('Оцінка ASRI-Audio на незалежному голосі lada:')
    display(hsummary)
    print('Парні довірчі інтервали (умовні для одного TTS-голосу):')
    display(diff_df)

## Інтерпретація для статті в *AI (MDPI)*

Після hidden test інтерпретуємо **тільки заздалегідь зафіксовані моделі, пороги та сценарії**. Різниця між clean і stress та зміни профілів TPR/TNR — головний результат сценарного тестування; оптимізувати модель після перегляду hidden **не можна**.

Повне дослідження охоплює 4 885 придатних канонічних AUA-записів та 513 REAL–TTS пар; незалежна статистика для matched-частини обмежена невеликою кількістю голосів. Навіть після успішного hidden test **генератор Piper залишається спільним**, тому переносимість на інші TTS-технології окремо не доведена.

Не змішуйте TRAIN/ADAPTATION зі статистично незалежними VALIDATION/HIDDEN. Не порівнюйте загальну BA змішаного AUA з matched-парами без застережень щодо джерела й класу.
